# Introduction to Restricted Boltzmann Machines
In this module, we will discuss Restricted Boltzmann Machines (RBMs), a generative stochastic neural network that can learn a probability distribution over its set of inputs. RBMs are a special case of Boltzmann Machines, which are undirected graphical models that can learn to represent complex distributions over their inputs.

By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:

* __Restricted Boltzmann Machines__ (RBMs) are a class of _generative_ stochastic neural networks. More specifically, given some (binary) input data $\mathbf{x}\in\left\{-1,1\right\}^{n}$, an RBM can be trained to approximate the probability distribution of this input. Moreover, once the RBM is trained to approximate the input distribution, we can _sample_ from the network; in other words, we generate new instances from the learned probability distribution.
* __Bipartite graph structure__. RBMs have [a bipartite graph structure](https://en.wikipedia.org/wiki/Bipartite_graph). The first layer is the _visible_ layer, while the second is the _hidden_ layer. A set of weighted edges connects the two layers, but there are no connections between the visible or hidden units in a later. This makes RBMs _restricted_ compared to general Boltzmann Machines, which can have connections between all units. In RBMs, the connections are only between the visible and hidden layers.
* __Training of RBMs__. RBMs are trained by maximizing the log likelihood of observing the data given the parameters, which is the same idea as the general Boltzmann machine. However, the bipartite structure allows an approximation of the likelihood gradient (using a concept called contrastive divergence), making training RBMs a tractable problem.


This is a super interesting topic, and we will explore it in detail. We will start with the basics of Boltzmann machines and discuss how we train them. Let's dive in!

___

<div>
    <center>
      <img
        src="figs/Fig-DrawSomethingLikeThis-but-Better-Redraw.png"
        alt="triangle with all three sides equal"
        height="600"
        width="300"
      />
    </center>
  </div>

## Restricted Boltzmann Machines (RBMs)
A restricted Boltzmann machine (RBM) is a type of Boltzmann machine that has a bipartite structure, meaning that the nodes can be divided into two disjoint sets: visible nodes and hidden nodes. The visible nodes represent the input data, while the hidden nodes capture the underlying structure of the data (latent variables).
> __How is this different from a Boltzmann machine?__ In a Boltzmann machine, we also have visible and hidden nodes, but all nodes are fully connected to each other, meaning that every node can influence every other node. In an RBM, the visible nodes are only connected to the hidden nodes, and the hidden nodes are only connected to the visible nodes. This means that there are no connections between the visible nodes or between the hidden nodes.

> __Why is this useful?__ The bipartite structure of RBMs allows for efficient training using contrastive divergence, as the number of connections between the visible and hidden nodes is much smaller than in a fully connected Boltzmann machine. This makes RBMs more practical for large datasets and complex models.

Let's look at the formal definition of an RBM. A restricted Boltzmann machine is defined by the tuple $\mathcal{B}_{\text{RBM}} = \left(\mathcal{V}_{v},\mathcal{V}_{h},\mathcal{E}, \mathbf{W},\mathbf{b}_{\text{v}}, \mathbf{b}_{\text{h}}, \mathbf{s}\right)$:
* __Visible Nodes__: The set of visible nodes $\mathcal{V}_{v}$ represents the input data. Each visible node $v_{i}\in\mathcal{V}_{v}$ has a binary state $s_{i}\in\{-1,1\}$ and a bias term $b_{i}\in\mathbf{b}_{v}$. There are $V = |\mathcal{V}_{v}|$ visible nodes.
* __Hidden Nodes__: The set of hidden nodes $\mathcal{V}_{h}$ captures the underlying structure of the data. Each hidden node $h_{j}\in\mathcal{V}_{h}$ has a binary state $s_{j}\in\{-1,1\}$ and a bias term $b_{j}\in\mathbf{b}_{h}$. There are $H = |\mathcal{V}_{h}|$ hidden nodes.
* __Edges__: The set of edges $\mathcal{E}$ connects the visible nodes to the hidden nodes. Each edge $e_{ij}\in\mathcal{E}$ connects a visible node $v_{i}\in\mathcal{V}_{v}$ to a hidden node $v_{j}\in\mathcal{V}_{h}$. The weight of the edge connecting $v_{i}$ and $v_{j}$ is denoted by $w_{ij}\in\mathbf{W}$, where the weight matrix $\mathbf{W}\in\mathbb{R}^{V\times{H}}$ is symmetric, i.e. $w_{ij} = w_{ji}$ and $w_{ii} = 0$ (no self loops). The weights $w_{ij}\in\mathbb{R}$ determine the strength of the connection between nodes $i$ and $j$. 
* __States__: The state of the RBM $\mathcal{B}_{\text{RBM}}$ is represented by a binary vector $\mathbf{s}\in\mathbb{R}^{|\mathcal{V}_{v}| + |\mathcal{V}_{h}|}$, where $s_{i}\in\{-1,1\}$ is the state of node $v_{i}$ and $s_{j}\in\{-1,1\}$ is the state of node $h_{j}$. The set of all possible _state configurations_ is denoted by $\mathcal{S} \equiv \left\{\mathbf{s}^{(1)},\mathbf{s}^{(2)},\ldots,\mathbf{s}^{(N)}\right\}$, where $N$ is the number of possible state configurations, or $N = 2^{V+H}$ for binary units.


Now that have formally defined the machine $\mathcal{B}_{\text{RBM}}$, how can we use it, i.e., how can we draw samples from it?

## Restricted Boltzmann Dynamics
Suppose we let the state of a Restricted Boltzmann Machine $\mathcal{B}_{\text{RBM}}$ evolve over $t=1,2,\dots, T$ turns, where the state of each node at turn $t$ is binary $s_{i}^{(t)} \in \{-1, 1\}$. During each turn, every visible and hidden node can update its state based on the states of the nodes its connected to, the weights of its connections, and its bias term. 
> __Connections:__ In a Restricted Boltzmann Machine, visible nodes are only connected to the hidden nodes, and vice-versa, and there are no connections between the visible nodes or between the hidden nodes. Thus, there will be $V\times{H}$ connections in total, where $V$ is the number of visible nodes and $H$ is the number of hidden nodes.

Let the nodes in the RBM be denoted by $\mathcal{V} = \mathcal{V}_{v}\cup\mathcal{V}_{h}$, where $\mathcal{V}_{v}$ is the set of visible nodes and $\mathcal{V}_{h}$ is the set of hidden nodes. Then, the total input to hidden (visible) node $v_{i}$ at turn $t$ denoted as $I_{h,i}^{(t)}$ (and $I_{v,i}^{(t)}$) is given by:
$$
\begin{align*}
I_{\star,i}^{(t)} &= \sum_{j\in\mathcal{V}_{\lnot{\star}}} w_{ij}s_{\lnot{\star},j}^{(t-1)} + b_{\star,i}\quad\forall i\in\mathcal{V}_{\star}\quad\star \in \{v,h\} \\
\end{align*}
$$
where $w_{ij}$ is the weight of the edge connecting $v_{i}$ and $v_{j}$, and $s_{\star,j}^{(t-1)}$ is the state of node $v_{j}$ in layer $\star = \{v,h\}$ at turn $t-1$. Like a full Boltzmann Machine, the state of each node in a restricted Boltzmann Machine is updated stochastically. The probability that node $v_{i}$ is `on` at turn $t$ is given by the logistic function:
$$
\begin{align*}
P(s_{\star,i}^{(t)} = 1 \mid {s}_{\lnot{\star},j}) & = \frac{\exp\left(-\beta\,E(s_{\star,i} = 1 \mid {s}_{\lnot{\star},j})\right)}{\exp\left(-\beta\,E(s_{\star,i} = 1 \mid {s}_{\lnot{\star},j})\right) + \exp\left(-\beta\,E(s_{\star,i} = -1 \mid {s}_{\lnot{\star},j})\right)} \\
\end{align*}
$$
However, we can simplify these probability expressions by substituting in the energy function:
$$
\begin{align*}
E(s_{\star,i}^{(t)} = 1 \mid {s}_{\lnot{\star},j}) & = -s_{\star,i}^{(t)}I_{\star,i}^{(t)} \\
\end{align*}
$$
which gives:
$$
\begin{align*}
P(s_{\star,i}^{(t)} = 1 \mid s_{\lnot{\star},j}) & = \frac{\exp\left(\beta\,I_{\star,i}^{(t)}\right)}{\exp\left(\beta\,I_{\star,i}^{(t)}\right) + \exp\left(-\beta\,I_{\star,i}^{(t)}\right)} \\
& = \frac{1}{1 + \exp(-2\;\beta\,I_{\star,i}^{(t)})}\quad\star \in \{v,h\}\quad\blacksquare
\end{align*}
$$
where $P(s_{\star,i}^{(t)} = 1 \mid s_{\lnot{\star},j})$ is the probability that node $v_{i}$ is `on` at time $t$ given the state of all the nodes __not__ in layer $\star$ (which we write in compact form as $\lnot\star$). The probability that node $v_{i}$ is `off` at time $t$ is given by $P(s_{\star,i}^{(t)} = -1| s_{\lnot{\star},j}) = 1 - P(s_{\star,i}^{(t)} = 1 \mid s_{\lnot{\star},j})$,  i.e., one minus the probability that the node is `on`.
* _What is β_? The parameter $\beta$ is the (inverse) temperature parameter that controls the amount of randomness in the system. As $\beta\rightarrow\infty$, the Boltzmann Machine becomes more deterministic; however, as $\beta\rightarrow{0}$, the Boltzmann Machine becomes more random.
* _What is $s_{\lnot{\star},j}$?_ This notation refers to the state of all nodes in the network that are _not_ in layer $\star$. Notice we have not included a superscript $t$ on $s_{\lnot{\star},j}$. However, from the perspective of any node, the state of the system is always the state of the system at the previous turn, i.e., $s_{\lnot{\star},j} = \left\{s_{\lnot{\star},j}^{(t-1)}\right\}_{j\in\mathcal{V}_{\lnot{\star}}}$.

### Sampling a Restricted Boltzmann Machine (Gibbs Sampling)
To generate samples from a Restricted Boltzmann Machine, let us consider the following algorithm: 

__Initialize__ the weights $\mathbf{W}$ and biases $\mathbf{b}_{v}$ and $\mathbf{b}_{h}$ of the Restricted Boltzmann Machine. Provide an initial state $\mathbf{s}_{v}^{(0)}$ of the visible nodes, a system (inverse) temperature $\beta$, and the number of turns $T$ to run the sampling algorithm.

> **Choosing the Number of Sampling Steps (T)**: The choice of T depends on your specific application and desired balance between quality and computational cost:
>
> **For Sampling/Generation**:
> - **Burn-in period**: Use T = 100-1000 steps to allow the chain to reach equilibrium and forget the initial state
> - **Mixing time**: RBMs typically need 10-100 steps to mix well, depending on network size and temperature β
> - **Rule of thumb**: Start with T = 10 × (V + H) where V and H are the number of visible and hidden units
>
> **For Training (Contrastive Divergence)**:
> - **Short chains**: T = 1-10 steps (CD-1 to CD-10) are commonly used during training
> - **CD-1 is most popular**: Single step often sufficient for gradient estimation during training
>
> **Practical Considerations**:
> - **Higher β**: More deterministic systems may need fewer steps to converge
> - **Lower β**: More random systems need more steps to explore the distribution
> - **Network size**: Larger networks generally require more steps for thorough mixing
> - **Monitor convergence**: Track how state statistics change over time to assess if T is sufficient

For each turn $t=1,2,\dots,T$ __do__:
1. **Update hidden nodes**: For each hidden node $h_{j}\in\mathcal{V}_{h}$ __do__:
    - Compute the total input: $I_{h,j}^{(t)} \gets \sum_{i\in\mathcal{V}_{v}} w_{ij}s_{v,i}^{(t-1)} + b_{h,j}$
    - Compute activation probability: $P(s_{h,j}^{(t)} = 1 \mid \mathbf{s}_{v}^{(t-1)}) \gets \left(1+\exp(-2\beta{I}_{h,j}^{(t)})\right)^{-1}$
    - Sample: $s_{h,j}^{(t)} \sim \texttt{Bernoulli}(P(s_{h,j}^{(t)} = 1 \mid \mathbf{s}_{v}^{(t-1)}))$

2. **Update visible nodes**: For each visible node $v_{i}\in\mathcal{V}_{v}$ __do__:
    - Compute the total input: $I_{v,i}^{(t)} \gets \sum_{j\in\mathcal{V}_{h}} w_{ij}s_{h,j}^{(t)} + b_{v,i}$
    - Compute activation probability: $P(s_{v,i}^{(t)} = 1 \mid \mathbf{s}_{h}^{(t)}) \gets \left(1+\exp(-2\beta{I}_{v,i}^{(t)})\right)^{-1}$
    - Sample: $s_{v,i}^{(t)} \sim \texttt{Bernoulli}(P(s_{v,i}^{(t)} = 1 \mid \mathbf{s}_{h}^{(t)}))$

3. Store the state vectors $\mathbf{s}_{v}^{(t)}$ and $\mathbf{s}_{h}^{(t)}$ of the network at turn $t$.


### Convergence of Gibbs Sampling in RBMs

**Guaranteed Convergence**: The Gibbs sampling algorithm for RBMs is **guaranteed to converge** to the stationary (equilibrium) distribution of the RBM. This follows from the fact that the sampling process will eventually explore all possible states and settle into the correct probability distribution that the RBM represents.

**Convergence Rate**: The convergence behavior depends on the application context. For sampling and generation, the chain typically mixes well within 10-50 steps after a burn-in period of 100-1000 steps. For training using Contrastive Divergence, very short chains of 1-10 steps are commonly used. Larger networks require more steps for thorough mixing, while very large weights can create "sticky" states that slow convergence. The choice of initial state and network architecture (ratio of hidden to visible units) also significantly affects convergence speed.
___

## Training Restricted Boltzmann Machines
Training an RBM involves adjusting the weights and biases to maximize the likelihood of the observed data. However, computing the exact likelihood gradient is intractable due to the partition function, so we use an approximation called **contrastive divergence**.

### The Learning Problem
Given a dataset $\mathcal{D} = \{\mathbf{v}^{(1)}, \mathbf{v}^{(2)}, \ldots, \mathbf{v}^{(N)}\}$ of binary visible vectors, we want to find parameters $\mathbf{W}$, $\mathbf{b}_v$, and $\mathbf{b}_h$ that maximize the log-likelihood:

$$
\mathcal{L}(\mathbf{W}, \mathbf{b}_v, \mathbf{b}_h) = \sum_{n=1}^{N} \log P(\mathbf{v}^{(n)} \mid \mathbf{W}, \mathbf{b}_v, \mathbf{b}_h)
$$

The probability of a visible vector $\mathbf{v}$ is given by marginalizing over all hidden states:

$$
P(\mathbf{v}) = \frac{1}{Z} \sum_{\mathbf{h}} \exp(-E(\mathbf{v}, \mathbf{h}))
$$

where $Z$ is the partition function and $E(\mathbf{v}, \mathbf{h})$ is the energy function:

$$
E(\mathbf{v}, \mathbf{h}) = -\sum_{i,j} w_{ij} v_i h_j - \sum_i b_{v,i} v_i - \sum_j b_{h,j} h_j
$$

### Contrastive Divergence (CD-k)
The exact gradient computation requires computing expectations over the entire model distribution, which is intractable. Contrastive divergence approximates this by using a short Markov chain starting from the data.

**CD-k Algorithm** (where k is the number of Gibbs sampling steps):

__Initialize__ parameters $\mathbf{W}$, $\mathbf{b}_v$, $\mathbf{b}_h$, learning rate $\alpha$, and number of CD steps $k$.

For each training example $\mathbf{v}^{(n)}$ in the dataset __do__:

1. **Positive phase**: Set $\mathbf{v}^{(0)} = \mathbf{v}^{(n)}$ (clamp data to visible units)
   - Compute $P(\mathbf{h}^{(0)} = 1 \mid \mathbf{v}^{(0)})$ and sample $\mathbf{h}^{(0)}$

2. **Negative phase**: Run k steps of Gibbs sampling starting from $(\mathbf{v}^{(0)}, \mathbf{h}^{(0)})$:
   - For $t = 1, 2, \ldots, k$:
     - Sample $\mathbf{v}^{(t)} \sim P(\mathbf{v} \mid \mathbf{h}^{(t-1)})$
     - Sample $\mathbf{h}^{(t)} \sim P(\mathbf{h} \mid \mathbf{v}^{(t)})$

3. **Parameter updates**:
   - $\Delta w_{ij} = \alpha \left[ \langle v_i h_j \rangle_{\text{data}} - \langle v_i h_j \rangle_{\text{model}} \right]$
   - $\Delta b_{v,i} = \alpha \left[ \langle v_i \rangle_{\text{data}} - \langle v_i \rangle_{\text{model}} \right]$
   - $\Delta b_{h,j} = \alpha \left[ \langle h_j \rangle_{\text{data}} - \langle h_j \rangle_{\text{model}} \right]$

   where:
   - $\langle \cdot \rangle_{\text{data}}$ denotes expectations under the data distribution (positive phase)
   - $\langle \cdot \rangle_{\text{model}}$ denotes expectations under the model distribution (negative phase)

### Key Training Insights

**Why CD Works**: Even though CD is an approximation, it provides a good gradient estimate because:
- The positive phase captures what the model should do with real data
- The negative phase captures what the model actually does
- The difference drives learning toward the data distribution

**Common Hyperparameters**:
- **CD steps**: Usually $k = 1$ (CD-1) works well in practice
- **Learning rate**: Typically $\alpha = 0.01$ to $0.1$
- **Batch size**: Mini-batches of 10-100 examples
- **Hidden units**: Often 2-10 times the number of visible units

**Training Challenges**:
- **Mode collapse**: RBM may focus on only some modes of the data
- **Slow mixing**: Gibbs chain may not explore the full distribution
- **Hyperparameter sensitivity**: Learning rate and architecture choices matter significantly

___